<h1 style="font-size:40px;"> <center> IMPORTS </center> </h1>

In [ ]:
import numpy as np
import pandas as pd
from numba import njit
import vectorbt as vbt
import optuna
from functools import partial 
from optuna.logging import _get_library_root_logger

<h1 style="font-size:40px;"> <center> GETTING PRICE DATA </center> </h1>

In [ ]:
price_data = vbt.YFData.download(
    "AAPL", missing_index="drop", interval="1d"
)

price_close = price_data.get("Close")
price_close

<h1 style="font-size:40px;"> <center> CREATING SIGNAL </center> </h1>

In [ ]:
fast_ma = vbt.MA.run(price_close, 10, short_name="fast MA")
slow_ma = vbt.MA.run(price_close, 50, short_name="slow MA")
rsi = vbt.RSI.run(price_close)

long_entries = fast_ma.ma_crossed_above(slow_ma) & rsi.rsi_above(50)
short_entries = fast_ma.ma_crossed_below(slow_ma)  & rsi.rsi_below(50)

<h1 style="font-size:40px;"> <center> SIMPLE BACKTEST </center> </h1>

In [ ]:
pf = vbt.Portfolio.from_signals(
    price_close,
    init_cash=1000,
    size=np.inf, 
    fees=0.001,
    entries=long_entries,
    short_entries=short_entries,
    sl_stop=0.025,
    tp_stop=0.05,
    #adjust_sl_func_nb=adjust_sl_func_nb,
    #adjust_tp_func_nb=adjust_tp_func_nb,
    freq="1d",
    #upon_opposite_entry="close",
)

In [ ]:
pf.stats()

In [ ]:
metric = "total_return"
"""
metric = ("annualized_return",) 
metric = ("max_drawdown",)
metric = "positions.win_rate"
metric = "sharpe_ratio"
"""
pf_perf = pf.deep_getattr(metric)
pf_perf

In [ ]:
fig = pf.plot(subplots = ['cum_returns', 'orders', 'trade_pnl', 'cash', 'assets', 'value'])
fig.show()

In [ ]:
pf.positions.records_readable

<h1 style="font-size:30px;"> <center> WINDOW OPTIMIZATION </center> </h1>

In [ ]:
windows = np.arange(2, 101)
fast_ma, slow_ma = vbt.MA.run_combs(price_close, window=windows, r=2, short_names=['fast', 'slow'])
entries = fast_ma.ma_crossed_above(slow_ma)
exit_entries = fast_ma.ma_crossed_below(slow_ma)

In [ ]:
pf = vbt.Portfolio.from_signals(
    price_close,
    init_cash=1000,
    size=np.inf, 
    fees=0.001,
    entries=entries,
    short_entries=exit_entries,
    sl_stop=0.025,
    tp_stop=0.05,
    freq="1d",
)

In [ ]:
pf_perf = pf.deep_getattr("total_return")
pf_perf

In [ ]:
pf_perf_matrix = pf_perf.vbt.unstack_to_df(
        index_levels = "fast_window",
        column_levels = "slow_window")
pf_perf_matrix

In [ ]:
fig = pf.total_return().vbt.heatmap(
    x_level='fast_window', y_level='slow_window', symmetric=True,
    trace_kwargs=dict(colorbar=dict(title='Total return', tickformat='%')))
fig.show()

<h1 style="font-size:30px;"> <center> TRAILING SL & TP </center> </h1>

In [ ]:
@njit
def adjust_sl_func_nb(c):
    # Keep current settings until a finite entry price and active position exist.
    if (c.position_now == 0 or not np.isfinite(c.position_now)
            or not np.isfinite(c.init_price) or c.init_price <= 0
            or not np.isfinite(c.val_price_now)):
        return c.curr_stop, c.curr_trail
    current_profit = (c.val_price_now - c.init_price) / c.init_price
    if c.position_now < 0:
        current_profit = -current_profit
    if not np.isfinite(current_profit):
        return c.curr_stop, c.curr_trail
    if current_profit >= 0.40:
        return 0.25, True
    elif current_profit >= 0.25:
        return 0.15, True
    elif current_profit >= 0.20:
        return 0.07, True
    return c.curr_stop, c.curr_trail


In [ ]:
@njit
def adjust_tp_func_nb(c):
    if (c.position_now == 0 or not np.isfinite(c.position_now)
            or not np.isfinite(c.init_price) or c.init_price <= 0
            or not np.isfinite(c.val_price_now)):
        return c.curr_stop
    current_profit = (c.val_price_now - c.init_price) / c.init_price
    if c.position_now < 0:
        current_profit = -current_profit
    if not np.isfinite(current_profit):
        return c.curr_stop
    if current_profit >= 0.50:
        return 0.55
    return c.curr_stop


In [ ]:
pf = vbt.Portfolio.from_signals(
    price_close,
    init_cash=1000,
    size=1, 
    fees=0,
    entries=long_entries,
    short_entries=short_entries,
    #sl_stop=0.025,
    adjust_sl_func_nb=adjust_sl_func_nb,
    adjust_tp_func_nb=adjust_tp_func_nb,
    #tp_stop=0.05,
    freq="1d",
    #upon_opposite_entry="close",
)

In [ ]:
pf_perf = pf.deep_getattr("total_return")
pf_perf

In [ ]:
pf.positions.records_readable

<h1 style="font-size:30px;"> <center> FPE BOX </center> </h1>

In [ ]:
def apply_margin(prices: np.array, signals:np.array, margin: float, short: bool = False):
    i = 0
    margin = -1*margin if short else margin
    while i < len(signals):
        if signals[i]:          
            init_price = prices[i]
            entry_price = init_price * (1+margin)
            exit_price = init_price * (1-margin)
            j = i
            while j < len(signals):
                signals[j] = False
                if short:
                    if prices[j] <= entry_price:
                        signals[j] = True
                        i = j
                        break
                    elif prices[j] >= exit_price:
                        signals[j] = False
                        i = j
                        break
                    
                else:
                    if prices[j] >= entry_price:
                        signals[j] = True
                        i = j
                        break
                    elif prices[j] <= exit_price:
                        signals[j] = False
                        i = j
                        break
                j += 1
        i += 1    

In [ ]:
# apply_margin mutates signals; each timed iteration needs the same original entries.
%timeit apply_margin(price_close.values, long_entries.values.copy(), margin=0.01)


<h1 style="font-size:30px;"> <center> GRID SEARCH </center> </h1>

In [ ]:
results = {}
prices = price_close.values

margin_range = list(np.arange(0, 0.05, 0.001))
for margin_val in margin_range:
    long_signals = long_entries.copy().values
    #short_signals = short_entries.copy().values
    apply_margin(prices, long_signals, margin=margin_val)
    #apply_margin(prices, short_signals, margin=margin_val, short=True)
    pf = vbt.Portfolio.from_signals(
    prices,
    init_cash=1000,
    #size=1, 
    fees=0.001,
    entries=long_signals,
    #short_entries=short_signals,
    sl_stop=0.025,
    tp_stop=0.05,
    #adjust_sl_func_nb=adjust_sl_func_nb,
    #adjust_tp_func_nb=adjust_tp_func_nb,
    freq="1d",
)
    
    pf_perf = pf.deep_getattr("total_return")
    results[margin_val] = pf_perf

In [ ]:
results

<h1 style="font-size:30px;"> <center> BAYESIAN SEARCH </center> </h1>

In [ ]:
prices = price_close.values

def objective(trial):
    params = {
        #'margin': trial.suggest_uniform('margin', 0.0001, 0.05),   
        'sl_stop': trial.suggest_uniform('sl_stop', 0.001, 0.05),   
        'tp_stop': trial.suggest_uniform('tp_stop', 0.001, 0.05),   
        }    
    long_signals = long_entries.copy().values
    #short_signals = short_entries.copy().values
    apply_margin(prices, long_signals, margin=0)
    #apply_margin(prices, short_signals, margin=margin_val, short=True)
    pf = vbt.Portfolio.from_signals(
    prices,
    init_cash=1000,
    #size=1, 
    fees=0.001,
    entries=long_signals,
    #short_entries=short_signals,
    sl_stop=params["sl_stop"],
    tp_stop=params["tp_stop"],
    #adjust_sl_func_nb=adjust_sl_func_nb,
    #adjust_tp_func_nb=adjust_tp_func_nb,
    freq="1d",
    )
    return pf.deep_getattr("total_return")

sampler = optuna.samplers.TPESampler(seed=42)
pruner=optuna.pruners.SuccessiveHalvingPruner()
study = optuna.create_study(
    direction="maximize",
    study_name="Bayesian Backtest Optimizer",
    sampler=sampler,
    pruner=pruner,
)
study.optimize(objective, n_trials=1000,)
best_trial = study.best_trial

print(f"Best Total Return: {best_trial.value}")
print(f"Best hyper parameters: {best_trial.params}")

<h1 style="font-size:30px;"> <center> OPTUNA EARLY STOPPING </center> </h1>

In [ ]:
logger = _get_library_root_logger()

def early_stopping_check(study, trial, early_stopping_rounds=10):
    current_trial_number = trial.number
    best_trial_number = study.best_trial.number
    should_stop = (current_trial_number - best_trial_number) >= early_stopping_rounds
    if should_stop:
        logger.info("Early stopping detected after last %s trial did not make any improvement", early_stopping_rounds)
        study.stop()
        

In [ ]:
prices = price_close.values

def objective(trial):
    params = {
        #'margin': trial.suggest_uniform('margin', 0.0001, 0.05),   
        'sl_stop': trial.suggest_uniform('sl_stop', 0.001, 0.1),   
        'tp_stop': trial.suggest_uniform('tp_stop', 0.001, 0.1),   
        }    
    long_signals = long_entries.copy().values
    #short_signals = short_entries.copy().values
    apply_margin(prices, long_signals, margin=0)
    #apply_margin(prices, short_signals, margin=margin_val, short=True)
    pf = vbt.Portfolio.from_signals(
    prices,
    init_cash=1000,
    #size=1, 
    fees=0.001,
    entries=long_signals,
    #short_entries=short_signals,
    sl_stop=params["sl_stop"],
    tp_stop=params["tp_stop"],
    #adjust_sl_func_nb=adjust_sl_func_nb,
    #adjust_tp_func_nb=adjust_tp_func_nb,
    freq="1d",
    )
    return pf.deep_getattr("total_return")

sampler = optuna.samplers.TPESampler(seed=42)
pruner=optuna.pruners.SuccessiveHalvingPruner()
study = optuna.create_study(
    direction="maximize",
    study_name="Bayesian Backtest Optimizer",
    sampler=sampler,
    pruner=pruner,
)

early_stopping_rounds = 100

study.optimize(objective, n_trials=1000, callbacks=[partial(early_stopping_check, early_stopping_rounds=early_stopping_rounds)],)
best_trial = study.best_trial

print(f"Best Total Return: {best_trial.value}")
print(f"Best hyper parameters: {best_trial.params}")